# Video Processing Pipeline - Team Template

## Overview
This notebook processes video links (Instagram, YouTube, etc.) to:
- Download videos
- Extract frames at key points (first frame, 1-second frame, scene changes)
- Generate transcriptions using AI (Whisper model)
- Store metadata and results in organized folders

## Setup Instructions
1. Run the first cell to set up the environment
2. In Cell 2, provide your video links
3. Run the remaining cells to process your videos

**Note:** This notebook creates an isolated virtual environment, so it won't affect your system Python installation.

---
## STEP 1: Environment Setup (Run this first!)

In [ ]:
import subprocess
import sys
import os
import venv
import site

# Setup: Create and activate a temporary virtual environment with all necessary libraries
venv_path = os.path.join(os.getcwd(), "processing_venv")

print("Setting up virtual environment and installing dependencies...\n")

# Create virtual environment
if not os.path.exists(venv_path):
    venv.create(venv_path, with_pip=True)
    print(f"✓ Virtual environment created at {venv_path}")
else:
    print(f"✓ Virtual environment already exists at {venv_path}")

# Determine the pip and python executable paths
if sys.platform == "win32":
    pip_executable = os.path.join(venv_path, "Scripts", "pip.exe")
    python_executable = os.path.join(venv_path, "Scripts", "python.exe")
    site_packages = os.path.join(venv_path, "Lib", "site-packages")
else:
    pip_executable = os.path.join(venv_path, "bin", "pip")
    python_executable = os.path.join(venv_path, "bin", "python")
    site_packages = os.path.join(venv_path, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")

# List of required packages
required_packages = [
    "yt-dlp",
    "pandas",
    "faster-whisper",
    "ffmpeg-python"
]

print("\nInstalling required packages...\n")
for package in required_packages:
    try:
        subprocess.run(
            [pip_executable, "install", "-q", package],
            check=True,
            capture_output=True
        )
        print(f"✓ {package} installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Failed to install {package}: {e}")

print("\n✓ All dependencies installed successfully!")
print(f"Virtual environment location: {venv_path}")

# Activate the virtual environment for this notebook
print("\nActivating virtual environment for notebook...")

# Add venv site-packages to sys.path if not already present
if site_packages not in sys.path:
    sys.path.insert(0, site_packages)
    print(f"✓ Added {site_packages} to Python path")

# Set environment variables to use the venv
os.environ["VIRTUAL_ENV"] = venv_path
os.environ["PATH"] = os.path.join(venv_path, "Scripts" if sys.platform == "win32" else "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"✓ Virtual environment activated: {venv_path}")

# Check and install FFmpeg if needed
print("\n" + "="*60)
print("Checking for FFmpeg installation...")
print("="*60 + "\n")

try:
    result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("✓ FFmpeg is already installed")
    else:
        raise Exception("FFmpeg not found")
except (FileNotFoundError, subprocess.TimeoutExpired, Exception):
    print("FFmpeg not found. Installing...\n")
    
    if sys.platform == "win32":
        # Try to install FFmpeg on Windows using chocolatey
        try:
            subprocess.run(["choco", "--version"], capture_output=True, check=True)
            print("  Installing FFmpeg via Chocolatey...")
            subprocess.run(["choco", "install", "-y", "ffmpeg"], check=False)
            print("✓ FFmpeg installed via Chocolatey")
        except:
            print("  ✗ Chocolatey not found. Please install FFmpeg manually:")
            print("    Option 1: Download from https://ffmpeg.org/download.html")
            print("    Option 2: Install Chocolatey first: https://chocolatey.org/install")
    elif sys.platform == "linux" or sys.platform == "linux2":
        # Linux: use apt-get
        print("  Installing FFmpeg via apt...")
        subprocess.run(["sudo", "apt-get", "update"], check=False, capture_output=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "ffmpeg"], check=False)
        print("✓ FFmpeg installed via apt")
    elif sys.platform == "darwin":
        # macOS: use brew
        print("  Installing FFmpeg via Homebrew...")
        subprocess.run(["brew", "install", "ffmpeg"], check=False)
        print("✓ FFmpeg installed via Homebrew")
    else:
        print(f"  ✗ Unknown OS: {sys.platform}. Please install FFmpeg manually.")

print("\n" + "="*60)
print("✓ Setup complete! Ready to process videos.")
print("="*60)

---
## STEP 2: Provide Your Video Links

**Replace the placeholder below with your video links.** You can provide them as:
- A list of URLs
- A pandas DataFrame column
- A CSV file
- Any other format

The important thing is to have `video_links` as a list or pandas Series at the end.

In [ ]:
import pandas as pd
import os

# ============================================================
# OPTION 1: Define video links directly as a list
# ============================================================
# Example:
# video_links = [
#     "https://www.instagram.com/p/ABC123/",
#     "https://www.instagram.com/p/XYZ789/",
# ]

# ============================================================
# OPTION 2: Load from CSV file
# ============================================================
# Example:
# df = pd.read_csv("your_dataset.csv")
# video_links = df['url_column_name']  # Or however your links are structured

# ============================================================
# OPTION 3: Load from any other format
# ============================================================
# (Add your custom loading code here)

# ============================================================
# TODO: REPLACE THIS WITH YOUR ACTUAL DATA
# ============================================================
video_links = [
    # "https://www.instagram.com/p/YOUR_VIDEO_1/",
    # "https://www.instagram.com/p/YOUR_VIDEO_2/",
]

print(f"Loaded {len(video_links)} video links")
if video_links:
    print("\nFirst few links:")
    for link in video_links[:5]:
        print(f"  - {link}")
else:
    print("\n⚠️  WARNING: No video links loaded! Please update the cell above before proceeding.")

---
## STEP 3: Load Whisper Model for Transcription

This loads the AI model that will transcribe audio from videos. It may take a moment on first run.

In [ ]:
try:
    from faster_whisper import WhisperModel
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "faster-whisper"], check=True)
    from faster_whisper import WhisperModel

import os

# Load model ONCE - this saves time and VRAM
print("Loading Whisper large-v3-turbo model into memory...")
print("(This may take 30-60 seconds on first run)\n")

model = WhisperModel("large-v3-turbo", device="cpu", compute_type="float32")
print("✓ Model loaded successfully!")

def transcribe_audio(video_path, output_folder):
    """Transcribes video audio and saves timestamped text to a file."""
    print(f"\nTranscribing audio for {os.path.basename(video_path)}...")

    # Transcribe the file
    # Language auto-detection is enabled (leaves language=None)
    # beam_size=5 improves accuracy
    segments, info = model.transcribe(video_path, beam_size=5)

    print(f"Detected language: '{info.language}' (confidence: {info.language_probability * 100:.1f}%)")

    # Save timestamped transcription
    transcript_path = os.path.join(output_folder, "transcription.txt")

    with open(transcript_path, "w", encoding="utf-8") as f:
        for segment in segments:
            # Format: [0.00s -> 4.50s] Text content
            line = f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}\n"
            f.write(line)
            print(line.strip())

    print(f"\n✓ Transcription saved: {transcript_path}")
    return transcript_path

---
## STEP 4: Define Frame Extraction Function

This uses FFmpeg to extract key frames from videos:
- First frame (at 0 seconds)
- Frame at 1 second
- Frames at scene changes (configurable threshold)

In [ ]:
import subprocess

# Configuration for frame extraction
SCENE_THRESHOLD = 0.4  # Adjust this to detect more or fewer scene changes (0.0 to 1.0)

def extract_frames(video_path, output_folder):
    """Uses FFmpeg to extract first frame, 1-second frame, and scene changes."""
    os.makedirs(output_folder, exist_ok=True)

    print(f"Extracting frames for {os.path.basename(video_path)}...")

    # 1. Extract the very first frame (0 seconds)
    first_frame_path = os.path.join(output_folder, "frame_00_first.jpg")
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-ss", "00:00:00", "-vframes", "1", "-q:v", "2", first_frame_path],
        check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    if os.path.exists(first_frame_path):
        print(f"  ✓ First frame extracted")

    # 2. Extract the frame at exactly 1 second
    second_frame_path = os.path.join(output_folder, "frame_01_second.jpg")
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-ss", "00:00:01", "-vframes", "1", "-q:v", "2", second_frame_path],
        check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    if os.path.exists(second_frame_path):
        print(f"  ✓ One-second frame extracted")

    # 3. Extract frames based on scene changes
    output_pattern = os.path.join(output_folder, "frame_%04d_scene.jpg")
    command = [
        "ffmpeg", "-y", "-i", video_path, "-vf", f"select='gt(scene,{SCENE_THRESHOLD})'",
        "-vsync", "vfr", "-q:v", "2", output_pattern
    ]
    try:
        result = subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"  ✓ Scene change frames extracted to {output_folder}")
    except subprocess.CalledProcessError as e:
        print(f"  ⚠️  Error extracting scene frames: {e}")

print(f"✓ Frame extraction function ready (scene threshold: {SCENE_THRESHOLD})")

---
## STEP 5: Main Processing Function

This function ties everything together:
1. Downloads the video
2. Extracts metadata
3. Extracts frames
4. Generates transcription
5. Saves all results in organized folders

In [ ]:
import subprocess
import shutil
try:
    import yt_dlp
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "yt-dlp"], check=True)
    import yt_dlp

import glob

# Configuration
DATASET_DIR = "multimodal_dataset"  # Folder where all downloaded videos and metadata will be stored
PROCESSED_URLS_FILE = "processed_urls.txt"  # Track which URLs have been processed

def process_video_url(url):
    """Downloads a video (Instagram, YouTube, etc.) and extracts all data.
    
    Returns:
        dict: Information about the processed video, or None if failed
    """
    print(f"\n{'='*70}")
    print(f"Processing: {url}")
    print(f"{'='*70}")
    
    ydl_opts = {
        'outtmpl': f'{DATASET_DIR}/%(id)s/%(id)s.%(ext)s',
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'writeinfojson': True,
        'sleep_interval_requests': 15,  # Be polite to servers
        'sleep_interval': 20,
        'max_sleep_interval': 50,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            print("  Downloading...")
            info_dict = ydl.extract_info(url, download=True)
            video_id = info_dict.get('id', 'unknown_id')
            print(f"  ✓ Downloaded with ID: {video_id}")

        video_folder = os.path.join(DATASET_DIR, video_id)
        
        # Check if metadata file exists (indicates successful download)
        info_json_path = os.path.join(video_folder, f"{video_id}.info.json")
        if not os.path.exists(info_json_path):
            print(f"  ✗ Could not find metadata file for {video_id}")
            return None

        print(f"  ✓ Metadata saved")

        # Look for the video file
        video_files = glob.glob(os.path.join(video_folder, "*.mp4")) + \
                     glob.glob(os.path.join(video_folder, "*.mkv")) + \
                     glob.glob(os.path.join(video_folder, "*.webm"))
        
        if not video_files:
            print(f"  ⚠️  No video file found (might be age-restricted or unavailable)")
            print(f"  Location: {video_folder}")
            return None

        video_path = video_files[0]
        print(f"  ✓ Video file: {os.path.basename(video_path)}")

        # Extract frames
        frames_folder = os.path.join(video_folder, "frames")
        extract_frames(video_path, frames_folder)

        # Transcribe audio
        transcribe_audio(video_path, video_folder)

        # Check for audio/music
        has_music = False
        if info_dict.get('acodec') and info_dict['acodec'] != 'none':
            has_music = True
        else:
            for f in info_dict.get('formats', []):
                if f.get('acodec') and f['acodec'] != 'none':
                    has_music = True
                    break

        music_title = info_dict.get('track') or info_dict.get('title')
        music_artist = info_dict.get('artist')

        result = {
            'url': url,
            'video_id': video_id,
            'video_folder': video_folder,
            'has_audio': has_music,
            'music_title': music_title if has_music else None,
            'music_artist': music_artist if has_music else None,
        }
        
        print(f"  ✓ Successfully processed!")
        return result
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return None

print("✓ Processing function ready")

---
## STEP 6: Process All Videos

Run this cell to start processing your videos. It will:
- Skip any videos that were already successfully processed
- Download new videos
- Extract frames and transcriptions
- Save everything in the `multimodal_dataset` folder

In [ ]:
import os

# Load previously processed URLs from local file (to avoid re-downloading)
if os.path.exists(PROCESSED_URLS_FILE):
    with open(PROCESSED_URLS_FILE, "r") as f:
        processed_urls = set(line.strip() for line in f)
else:
    processed_urls = set()

print(f"Previously processed URLs: {len(processed_urls)}")
print(f"New URLs to process: {len(set(video_links) - processed_urls)}\n")

# Create output directory
os.makedirs(DATASET_DIR, exist_ok=True)

collected_video_data = []
urls_to_process = [url for url in video_links if url not in processed_urls]

for idx, url in enumerate(urls_to_process, 1):
    print(f"\n[{idx}/{len(urls_to_process)}] Processing URL...")
    
    video_info = process_video_url(url)
    if video_info:
        collected_video_data.append(video_info)

        # Record the successfully processed URL
        with open(PROCESSED_URLS_FILE, "a") as f:
            f.write(f"{url}\n")
        processed_urls.add(url)

print(f"\n\n{'='*70}")
print(f"✓ Processing complete!")
print(f"Successfully processed: {len(collected_video_data)} videos")
print(f"Failed or skipped: {len(urls_to_process) - len(collected_video_data)} videos")
print(f"Output folder: {DATASET_DIR}")
print(f"{'='*70}")

---
## STEP 7: Validate Results

This cell checks which videos were successfully processed with all components (video, frames, transcription).

In [ ]:
import json
from pathlib import Path

DATASET_DIR = Path("multimodal_dataset")

urls = video_links if isinstance(video_links, list) else video_links.tolist()
subdirs = [p for p in DATASET_DIR.iterdir() if p.is_dir()] if DATASET_DIR.exists() else []

results = {
    "total_urls": len(urls),
    "not_downloaded": [],
    "downloaded_partial": [],
    "downloaded_complete": []
}

def folder_matches_url(folder: Path, url: str) -> bool:
    """Check if a folder corresponds to a given URL."""
    # Fast check: folder name in url (common for yt-dlp ids)
    if folder.name in url:
        return True
    # Otherwise inspect any .info.json files for matching webpage/original url
    for j in folder.glob("*.info.json"):
        try:
            data = json.loads(j.read_text(encoding="utf-8"))
        except Exception:
            continue
        for key in ("webpage_url", "original_url", "url"):
            val = data.get(key)
            if isinstance(val, str) and val == url:
                return True
        # Fallback: search any string fields for the url substring
        if any(isinstance(v, str) and url in v for v in data.values()):
            return True
    return False

for url in urls:
    match = None
    for folder in subdirs:
        if folder_matches_url(folder, url):
            match = folder
            break
    
    if match is None:
        results["not_downloaded"].append(url)
        continue

    info_jsons = list(match.glob("*.info.json"))
    transcript = match / "transcription.txt"
    frames_folder = match / "frames"
    video_files = list(match.glob("*.mp4")) + list(match.glob("*.mkv")) + list(match.glob("*.webm"))

    checks = {
        "metadata": bool(info_jsons),
        "transcription": transcript.exists(),
        "frames": frames_folder.exists() and any(frames_folder.iterdir()),
        "video": bool(video_files)
    }

    if all(checks.values()):
        results["downloaded_complete"].append({"url": url, "folder": str(match)})
    else:
        results["downloaded_partial"].append({"url": url, "folder": str(match), "checks": checks})

# Print report
print("\n" + "="*70)
print("VALIDATION REPORT")
print("="*70)
print(f"Total URLs: {results['total_urls']}")
print(f"✓ Complete (all components): {len(results['downloaded_complete'])}")
print(f"⚠️  Partial (missing components): {len(results['downloaded_partial'])}")
print(f"✗ Not downloaded: {len(results['not_downloaded'])}")

if results["downloaded_partial"]:
    print("\nPartially downloaded videos (showing first 5):")
    for e in results["downloaded_partial"][:5]:
        checks_str = ", ".join([f"{k}: {'✓' if v else '✗'}" for k, v in e['checks'].items()])
        print(f"  {e['url'][:50]}...")
        print(f"    Folder: {e['folder']}")
        print(f"    Status: {checks_str}")

if results["not_downloaded"]:
    print("\nNot downloaded (showing first 5):")
    for u in results["not_downloaded"][:5]:
        print(f"  {u[:60]}...")

print("\n" + "="*70)

---
## Notes & Troubleshooting

### Output Structure
Results are saved in `multimodal_dataset/` with the following structure:
```
multimodal_dataset/
├── VIDEO_ID_1/
│   ├── VIDEO_ID_1.mp4 (or .mkv/.webm)
│   ├── VIDEO_ID_1.info.json (metadata)
│   ├── transcription.txt (timestamped text)
│   └── frames/
│       ├── frame_00_first.jpg
│       ├── frame_01_second.jpg
│       └── frame_XXXX_scene.jpg (multiple)
├── VIDEO_ID_2/
│   └── ...
```

### Common Issues

**FFmpeg not found on Windows:**
- The script tries to install via Chocolatey
- If you don't have Chocolatey, install FFmpeg manually: https://ffmpeg.org/download.html
- Then add it to your PATH

**FFmpeg not found on Linux:**
- Run: `sudo apt-get install ffmpeg`

**FFmpeg not found on macOS:**
- Run: `brew install ffmpeg`

**Videos not downloading:**
- Some videos may be age-restricted, region-locked, or private
- Check the validation report to see which ones failed

**Slow processing:**
- Transcription is the slowest step (uses AI model)
- This is normal; be patient!

### Customization

**Adjust scene detection sensitivity:**
- In STEP 4, change `SCENE_THRESHOLD` (0.0 = very sensitive, 1.0 = not sensitive)

**Change output folder:**
- In STEP 5, modify `DATASET_DIR` variable

**Use different Whisper model:**
- In STEP 3, change model name: `"base"`, `"small"`, `"medium"`, `"large"`
- Larger = more accurate but slower and uses more memory